In [9]:
import os

os.environ["HADOOP_CONF_DIR"] = "/etc/hadoop/conf"
os.environ["YARN_CONF_DIR"] = "/etc/hadoop/conf"
os.environ["HIVE_CONF_DIR"] = "/etc/hive/conf"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("team17 create hive tables") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "/user/team17/project/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/05/06 16:57:53 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/05/06 16:57:53 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [10]:
spark

In [14]:
spark.conf.get("spark.sql.catalogImplementation")

'hive'

In [12]:
spark.sql("CREATE DATABASE IF NOT EXISTS team17_project")
spark.sql("USE team17_project")

26/05/06 16:58:09 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/06 16:58:09 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/06 16:58:14 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/06 16:58:14 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore team17@10.100.30.57
26/05/06 16:58:14 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/05/06 16:58:15 WARN ObjectStore: Failed to get database team17_project, returning NoSuchObjectException
26/05/06 16:58:15 WARN ObjectStore: Failed to get database team17_project, returning NoSuchObjectException
26/05/06 16:58:15 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/05/06 16:58:15 WARN ObjectStore: Failed to get database team17_pr

DataFrame[]

In [21]:
apps_path = "/user/team17/project/hive/warehouse/apps_part"

apps_df = spark.read.option("basePath", apps_path).parquet(apps_path)

apps_df.printSchema()
apps_df.show(5)

root
 |-- appid: integer (nullable = true)
 |-- userid: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- jobid: integer (nullable = true)
 |-- split: string (nullable = true)
 |-- windowid: integer (nullable = true)

+------+-------+-------------------+------+-----+--------+
| appid| userid|               date| jobid|split|windowid|
+------+-------+-------------------+------+-----+--------+
|353565|1471869|2012-04-01 14:20:09|655840|Train|       1|
|353564|1471869|2012-04-01 14:04:36|861987|Train|       1|
|353546|1471791|2012-04-05 14:18:40|766725|Train|       1|
|353414|1471266|2012-04-09 15:22:18|792301|Train|       1|
|353378|1470818|2012-04-04 10:24:20|324128|Train|       1|
+------+-------+-------------------+------+-----+--------+
only showing top 5 rows



In [23]:
# apps_part
spark.sql("DROP TABLE IF EXISTS team17_project.apps_part")

spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS team17_project.apps_part (
    appid INT,
    userid INT,
    date TIMESTAMP,
    jobid INT
)
PARTITIONED BY (
    split STRING,
    windowid INT
)
STORED AS PARQUET
LOCATION '/user/team17/project/hive/warehouse/apps_part'
""")

spark.sql("MSCK REPAIR TABLE team17_project.apps_part")

DataFrame[]

In [24]:
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM apps_part LIMIT 5").show()

+--------------+---------+-----------+
|     namespace|tableName|isTemporary|
+--------------+---------+-----------+
|team17_project|apps_part|      false|
+--------------+---------+-----------+

+------+-------+-------------------+------+-----+--------+
| appid| userid|               date| jobid|split|windowid|
+------+-------+-------------------+------+-----+--------+
|353565|1471869|2012-04-01 14:20:09|655840|Train|       1|
|353564|1471869|2012-04-01 14:04:36|861987|Train|       1|
|353546|1471791|2012-04-05 14:18:40|766725|Train|       1|
|353414|1471266|2012-04-09 15:22:18|792301|Train|       1|
|353378|1470818|2012-04-04 10:24:20|324128|Train|       1|
+------+-------+-------------------+------+-----+--------+



In [25]:
jobs_part = "/user/team17/project/hive/warehouse/jobs_part"

jobs_part_df = spark.read.option("basePath", jobs_part).parquet(jobs_part)

jobs_part_df.printSchema()
jobs_part_df.show(5)

root
 |-- jobid: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- requirements: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- start: timestamp (nullable = true)
 |-- finish: timestamp (nullable = true)
 |-- windowid: integer (nullable = true)

+------+--------------------+--------------------+--------------------+----------------+-----+-------+-------+-------------------+-------------------+--------+
| jobid|               title|         description|        requirements|            city|state|country|zipcode|              start|             finish|windowid|
+------+--------------------+--------------------+--------------------+----------------+-----+-------+-------+-------------------+-------------------+--------+
|895739|Outside Sales Rep...|<p><b><span>Outsi...|<p><b><span>Posit...|        

In [26]:
# jobs_part 
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS jobs_part (
    jobid INT,
    title STRING,
    description STRING,
    requirements STRING,
    city STRING,
    state STRING,
    country STRING,
    zipcode STRING,
    start TIMESTAMP,
    finish TIMESTAMP
)
PARTITIONED BY (
    windowid INT
)
STORED AS PARQUET
LOCATION '/user/team17/project/hive/warehouse/jobs_part'
""")

spark.sql("MSCK REPAIR TABLE team17_project.jobs_part")

DataFrame[]

In [27]:
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM apps_part LIMIT 5").show()

+--------------+---------+-----------+
|     namespace|tableName|isTemporary|
+--------------+---------+-----------+
|team17_project|apps_part|      false|
|team17_project|jobs_part|      false|
+--------------+---------+-----------+

+------+-------+-------------------+------+-----+--------+
| appid| userid|               date| jobid|split|windowid|
+------+-------+-------------------+------+-----+--------+
|353565|1471869|2012-04-01 14:20:09|655840|Train|       1|
|353564|1471869|2012-04-01 14:04:36|861987|Train|       1|
|353546|1471791|2012-04-05 14:18:40|766725|Train|       1|
|353414|1471266|2012-04-09 15:22:18|792301|Train|       1|
|353378|1470818|2012-04-04 10:24:20|324128|Train|       1|
+------+-------+-------------------+------+-----+--------+



In [34]:
users_part = "/user/team17/project/hive/warehouse/users_part"
users_df = spark.read.option("basePath", users_part).parquet(users_part)
users_df.printSchema()

root
 |-- userid: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- degree: string (nullable = true)
 |-- major: string (nullable = true)
 |-- graddate: timestamp (nullable = true)
 |-- jobcnt: integer (nullable = true)
 |-- yearsexp: integer (nullable = true)
 |-- employed: boolean (nullable = true)
 |-- managerexp: boolean (nullable = true)
 |-- emplcnt: integer (nullable = true)
 |-- split: string (nullable = true)
 |-- windowid: integer (nullable = true)



In [40]:
spark.sql("DROP TABLE IF EXISTS team17_project.users_part")

spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS team17_project.users_part (
    userid INT,
    city STRING,
    state STRING,
    country STRING,
    zipcode STRING,
    degree STRING,
    major STRING,
    graddate TIMESTAMP,
    jobcnt INT,
    yearsexp INT,
    employed BOOLEAN,
    managerexp BOOLEAN,
    emplcnt INT
)
PARTITIONED BY (
    split STRING,
    windowid INT
)
STORED AS PARQUET
LOCATION '/user/team17/project/hive/warehouse/users_part'
""")

spark.sql("MSCK REPAIR TABLE team17_project.users_part")

DataFrame[]

In [41]:
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM users_part LIMIT 5").show()

+--------------+-----------------+-----------+
|     namespace|        tableName|isTemporary|
+--------------+-----------------+-----------+
|team17_project|        apps_part|      false|
|team17_project|        jobs_part|      false|
|team17_project|user_history_part|      false|
|team17_project|       users_part|      false|
+--------------+-----------------+-----------+

+-------+-----------+-----+-------+-------+-----------+--------------------+-------------------+------+--------+--------+----------+-------+-----+--------+
| userid|       city|state|country|zipcode|     degree|               major|           graddate|jobcnt|yearsexp|employed|managerexp|emplcnt|split|windowid|
+-------+-----------+-----+-------+-------+-----------+--------------------+-------------------+------+--------+--------+----------+-------+-----+--------+
|1471960|    Phoenix|   AZ|     US|  85040|Associate's|             Nursing|2006-05-01 00:00:00|     9|      14|    true|     false|      0|Train|       1|

In [33]:
user_history = "/user/team17/project/hive/warehouse/user_history_part"
user_history_df = spark.read.option("basePath", user_history).parquet(user_history)
user_history_df.printSchema()

root
 |-- userid: integer (nullable = true)
 |-- sequence: integer (nullable = true)
 |-- jobtitle: string (nullable = true)
 |-- split: string (nullable = true)
 |-- windowid: integer (nullable = true)



In [36]:
spark.sql("DROP TABLE IF EXISTS team17_project.user_history_part")

spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS team17_project.user_history_part (
    userid INT,
    sequence INT,
    jobtitle STRING
)
PARTITIONED BY (
    split STRING,
    windowid INT
)
STORED AS PARQUET
LOCATION '/user/team17/project/hive/warehouse/user_history_part'
""")

spark.sql("MSCK REPAIR TABLE team17_project.user_history_part")

DataFrame[]

In [37]:
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM user_history_part LIMIT 5").show()

+--------------+-----------------+-----------+
|     namespace|        tableName|isTemporary|
+--------------+-----------------+-----------+
|team17_project|        apps_part|      false|
|team17_project|        jobs_part|      false|
|team17_project|user_history_part|      false|
|team17_project|       users_part|      false|
+--------------+-----------------+-----------+

+-------+--------+--------------------+-----+--------+
| userid|sequence|            jobtitle|split|windowid|
+-------+--------+--------------------+-----+--------+
|1471951|       4|Claims Resolution...|Train|       1|
|1471951|       3|Benefits Administ...|Train|       1|
|1471951|       2|Absence Managemen...|Train|       1|
|1471951|       1|Client Service An...|Train|       1|
|1471683|       6|Platoon leader an...|Train|       1|
+-------+--------+--------------------+-----+--------+

